# Portfolio Risk and Benchmark Dashboard

This project analyzes a stock portfolio using real market data. It calculates portfolio value, allocation, risk, and compares performance against the S&P 500.

## Import Libraries

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Portfolio Input

In [ ]:
portfolio = {
    "Ticker": ["AAPL", "MSFT", "NVDA"],
    "Shares": [10, 5, 3],
    "Purchase Price": [180, 420, 120]
}

portfolio_df = pd.DataFrame(portfolio)
portfolio_df

## Project Settings

In [ ]:
benchmark_ticker = "SPY"
risk_free_rate = 0.02
trading_days = 252

start_date = "2023-01-01"
end_date = "2026-06-25"

## Data Collection

In [ ]:
tickers = portfolio_df["Ticker"].tolist()
all_tickers = tickers + [benchmark_ticker]

data = yf.download(all_tickers, start=start_date, end=end_date, auto_adjust=True)

prices = data["Close"]
prices = prices.dropna()

prices.tail()

## Portfolio Value Calculation

In [ ]:
current_prices = prices[tickers].iloc[-1]

portfolio_df["Current Price"] = portfolio_df["Ticker"].map(current_prices)
portfolio_df["Current Value"] = portfolio_df["Shares"] * portfolio_df["Current Price"]
portfolio_df["Cost Basis"] = portfolio_df["Shares"] * portfolio_df["Purchase Price"]
portfolio_df["Profit/Loss"] = portfolio_df["Current Value"] - portfolio_df["Cost Basis"]
portfolio_df["Return %"] = portfolio_df["Profit/Loss"] / portfolio_df["Cost Basis"]

portfolio_df

## Portfolio Summary

In [ ]:
total_value = portfolio_df["Current Value"].sum()
total_cost = portfolio_df["Cost Basis"].sum()
total_profit_loss = portfolio_df["Profit/Loss"].sum()
total_return = total_profit_loss / total_cost

print("--- Portfolio Summary ---")
print(f"Current Value: ${round(total_value, 2)}")
print(f"Cost Basis: ${round(total_cost, 2)}")
print(f"Profit/Loss: ${round(total_profit_loss, 2)}")
print(f"Total Return: {round(total_return * 100, 2)}%")

## Allocation Analysis

In [ ]:
portfolio_df["Weight"] = portfolio_df["Current Value"] / total_value

largest_holding = portfolio_df.loc[portfolio_df["Weight"].idxmax()]

print("--- Allocation Analysis ---")

for i in range(len(portfolio_df)):
    ticker = portfolio_df.loc[i, "Ticker"]
    weight = portfolio_df.loc[i, "Weight"]
    print(f"{ticker}: {round(weight * 100, 2)}%")

print(f"\nLargest holding: {largest_holding['Ticker']}")
print(f"Largest holding weight: {round(largest_holding['Weight'] * 100, 2)}%")

## Risk Metrics

In [ ]:
stock_returns = prices[tickers].pct_change().dropna()

weights = portfolio_df["Weight"].values
portfolio_returns = stock_returns.dot(weights)

annual_return = (1 + portfolio_returns.mean()) ** trading_days - 1
volatility = portfolio_returns.std() * np.sqrt(trading_days)
sharpe_ratio = (annual_return - risk_free_rate) / volatility

portfolio_growth = (1 + portfolio_returns).cumprod()
running_max = portfolio_growth.cummax()
drawdown = portfolio_growth / running_max - 1
max_drawdown = drawdown.min()

print("--- Risk Metrics ---")
print(f"Annual Return: {round(annual_return * 100, 2)}%")
print(f"Volatility: {round(volatility * 100, 2)}%")
print(f"Sharpe Ratio: {round(sharpe_ratio, 2)}")
print(f"Maximum Drawdown: {round(max_drawdown * 100, 2)}%")

## Benchmark Comparison

In [ ]:
benchmark_returns = prices[benchmark_ticker].pct_change().dropna()
benchmark_returns = benchmark_returns.loc[portfolio_returns.index]

benchmark_annual_return = (1 + benchmark_returns.mean()) ** trading_days - 1
benchmark_volatility = benchmark_returns.std() * np.sqrt(trading_days)
relative_performance = annual_return - benchmark_annual_return

print("--- Benchmark Comparison ---")
print(f"Portfolio Annual Return: {round(annual_return * 100, 2)}%")
print(f"SPY Annual Return: {round(benchmark_annual_return * 100, 2)}%")
print(f"Relative Performance: {round(relative_performance * 100, 2)}%")

if annual_return > benchmark_annual_return:
    print("The portfolio outperformed SPY.")
else:
    print("The portfolio underperformed SPY.")

if volatility > benchmark_volatility:
    print("The portfolio took more risk than SPY.")
else:
    print("The portfolio took less risk than SPY.")

## Portfolio Growth Visualization

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(portfolio_growth, label="Portfolio")
plt.title("Portfolio Growth")
plt.ylabel("Growth of $1")
plt.legend()
plt.show()

## Allocation Visualization

In [ ]:
plt.figure(figsize=(7, 7))
plt.pie(portfolio_df["Weight"], labels=portfolio_df["Ticker"], autopct="%1.1f%%")
plt.title("Portfolio Allocation")
plt.show()

## Drawdown Visualization

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(drawdown, label="Drawdown", color="red")
plt.title("Portfolio Drawdown")
plt.ylabel("Drawdown")
plt.legend()
plt.show()

## Portfolio vs Benchmark Visualization

In [ ]:
benchmark_growth = (1 + benchmark_returns).cumprod()

plt.figure(figsize=(12, 6))
plt.plot(portfolio_growth, label="Portfolio")
plt.plot(benchmark_growth, label="SPY")
plt.title("Portfolio vs Benchmark")
plt.ylabel("Growth of $1")
plt.legend()
plt.show()

## Conclusion

This project shows how to analyze a portfolio using Python and real market data.

It calculates portfolio return, allocation, volatility, Sharpe Ratio, maximum drawdown, and compares the portfolio against SPY.

The main learning is that portfolio performance should be judged by both return and risk.